In [1]:
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset
import pandas as pd
import ast
from tqdm import tqdm
from torchvision import transforms
from torch.utils.data import DataLoader

In [2]:
import nbimporter
from prelimtrain import EmoVAE

ModuleNotFoundError: No module named 'prelimtrain'

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_DIM = 800 #128 note events  * 4 features
H_DIM = 200 
Z_DIM = 25 
CONDITION_DIM = 19 # 7 features , 12 music keys
NUM_EPOCHS =100
BATCH_SIZE =32

print("chicken")

model = EmoVAE(input_dim =INPUT_DIM, condition_dim = CONDITION_DIM, hidden_dim = H_DIM, z_dim = Z_DIM).to(DEVICE)


train = torch.load("bcvae_emopia.pt", map_location = DEVICE)

# Check what dimensions the saved model actually has
print(type(train))
if isinstance(train, dict):
    print(train.keys())

ckpt = torch.load("bcvae_emopia.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])

GLOBAL_MAX_START = train["global_max_start"]
GLOBAL_MAX_DURATION = train["global_max_duration"]
print(GLOBAL_MAX_START)
print(GLOBAL_MAX_DURATION)
model.eval()  #generate meeting
print("saved value with space:", train.get("global_max_start "))
print("saved value no space:", train.get("global_max_start"))
z = torch.randn(1, Z_DIM).to(DEVICE)
def generation_condition(batch, keyname="C"):

     c = torch.tensor(float(batch["arousal"]), 
                         float(batch["valence"]), 
      
                         dtype=torch.float32)
     return c
def generate_batch (model, batch, z, device, max_notes = 1000):
     with torch.no_grad():
          c = generation_condition(batch).unsqueeze(0).to(device)
          generate = model.decode(z,c)
          generate = generate.view(max_notes, 4)
          generate[:,0] = generate[:,0].clamp(0, 1)  # pitch
          generate[:,3] = generate[:,3].clamp(0, 1)  # velocity
        
          return generate.cpu()

chicken
<class 'dict'>
dict_keys(['model_state_dict', 'global_max_start', 'global_max_duration'])
80020.0
4612.0
saved value with space: None
saved value no space: 80020.0


In [ ]:
# history is the list of emotions batches over time musi generation running
def generate_song(model, emo_history, device, max_notes =1000, segment_ms = 10000):
    all_notes = []
    time_offset = 0.0

    for batch in emo_history:
        z = torch.randn(1, Z_DIM).to(device)
        notes = generate_batch(model, batch, z, device, max_notes)

        notes[:, 0] = notes[:, 0] * 127
        notes[:, 1] = notes[:, 1] * GLOBAL_MAX_START
        notes[:, 2] = notes[:, 2] * GLOBAL_MAX_DURATION
        notes[:, 3] = notes[:, 3] * 127

        notes[:, 1] = notes[:, 1] + time_offset

        segment_end = time_offset + segment_ms

        # keep notes that begin inside this segment
        notes = notes[notes[:, 1] < segment_end]

        # clip notes so they do not extend forever past the segment
        end_times = notes[:, 1] + notes[:, 2]
        clipped_end_times = torch.clamp(end_times, max=segment_end)
        notes[:, 2] = clipped_end_times - notes[:, 1]

        # optional: remove notes with zero or negative duration after clipping
        notes = notes[notes[:, 2] > 0]

        all_notes.append(notes)

        # move to next emotional segment no matter what
        time_offset += segment_ms 

    return all_notes
    

In [ ]:
import json
with open("emotion_batches.json", "r") as f:
    emo_batches = json.load(f)
z = torch.randn(1,Z_DIM).to(DEVICE)
all_generated = []
for i, batch in enumerate(emo_batches):
    z = torch.randn(1, Z_DIM).to(DEVICE)
    generated_notes = generate_batch(model, batch, z, DEVICE)
    print("Emotion", batch)
    all_generated.append(generated_notes)
print(all_generated)

Emotion {'start_batch': 1774238481.731126, 'end_batch': 1774238484.2638643, 'arousal': 0.37, 'valence': 0.88, 'intensity': 0.9, 'emotion_name': 'Extremely happy'}
Emotion {'start_batch': 1774238484.2638643, 'end_batch': 1774238486.2792518, 'arousal': 0.22, 'valence': -0.27, 'intensity': 0.35, 'emotion_name': 'Moderately annoyed'}
Emotion {'start_batch': 1774238486.2792518, 'end_batch': 1774238488.326993, 'arousal': 0.16, 'valence': -0.4, 'intensity': 0.44, 'emotion_name': 'Moderately frustrated'}
Emotion {'start_batch': 1774238488.326993, 'end_batch': 1774238490.385844, 'arousal': 0.18, 'valence': -0.43, 'intensity': 0.47, 'emotion_name': 'Moderately frustrated'}
Emotion {'start_batch': 1774238490.385844, 'end_batch': 1774238492.4961317, 'arousal': 0.29, 'valence': -0.21, 'intensity': 0.35, 'emotion_name': 'Moderately afraid'}
Emotion {'start_batch': 1774238492.4961317, 'end_batch': 1774238494.577863, 'arousal': 0.26, 'valence': -0.05, 'intensity': 0.27, 'emotion_name': 'Slightly arous

In [14]:
#Getting back intial values for music generation
unnormalized_gen = []
for g in all_generated:
    gen = g.clone()
    gen[:,0] = gen[:,0]*127.0 
    gen[:,3 ] = gen[:,3]*127.0
    gen[:,1] = gen[:,1]*GLOBAL_MAX_START
    gen[:,2] = gen[:,2]*GLOBAL_MAX_DURATION
    unnormalized_gen.append(gen)
    print( gen[:,1])

tensor([39947.5781, 39973.1367, 40116.7461, 39953.3086, 39812.3359, 40087.6484,
        39901.0938, 39986.5977, 39971.3633, 39713.5508, 39845.9180, 39751.0312,
        40097.2539, 39647.1211, 39372.1133, 39577.6680, 39575.3008, 39595.9727,
        39555.8320, 39692.3281, 39525.5586, 39233.8359, 39503.8906, 39310.5039,
        39322.8906, 39482.0469, 39600.9297, 39433.8555, 39631.0742, 39251.4141,
        39252.3281, 39053.1562, 39793.1484, 39065.3555, 38891.8984, 39031.4180,
        39219.9180, 38995.3945, 39276.3281, 39531.0898, 39238.2188, 38841.4414,
        38908.3281, 38644.0820, 38789.2266, 38629.3008, 38857.5078, 38706.9336,
        38737.5820, 39045.4180, 39106.9297, 38970.4180, 38818.1211, 38635.8320,
        38842.8281, 38439.1055, 38093.3477, 38521.9219, 38303.1914, 38690.2852,
        38694.9531, 38774.0977, 38932.5312, 38876.7383, 38859.9297, 38993.9102,
        38790.7070, 38439.2383, 39007.5586, 38648.1289, 38676.8164, 38874.9688,
        38671.2891, 38852.3711, 38717.90

In [15]:
unnormalized_gen_list = [g.detach().cpu().tolist() for g in unnormalized_gen]

import json
with open("unnormalized_generation.json", "w") as f:
    json.dump(unnormalized_gen_list, f, indent=2)
